# Embeddings basics

[Module 02](README.md) · [Workshop home](../README.md) · [Endpoint reference](../docs/inference.md)

**10 minutes.** What an embedding is, how to get one, and the two limits that shape
every index you will ever build.

An embedding turns text into a list of numbers. Text with similar meaning lands in a
similar direction, which lets you rank documents by relevance without matching a
single keyword. That is the whole idea behind retrieval.

## Step 1: Connect

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

root = Path.cwd()
if not (root / ".env").is_file() and (root.parent / ".env").is_file():
    root = root.parent
if not (root / ".env").is_file():
    raise RuntimeError("No .env found. See docs/00_prework.md")

load_dotenv(root / ".env", override=True)
client = OpenAI(
    base_url=os.environ["AKAMAI_BASE_URL"].strip().rstrip("/"),
    api_key=os.environ["AKAMAI_API_KEY"].strip(),
    timeout=60.0,
)

EMBED_MODEL = "qwen3-embedding-4b"
CHAT_MODEL = "qwen3-8b"
MODULE_DIR = root / "02_Embeddings_and_RAG"
print("Ready.")

## Step 2: Embed some text

`input` takes one string or a list of them. Each item in the response carries a
vector, in the same order you sent them.

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "A feline rested on a rug.",
    "Quarterly revenue grew eight percent.",
]

response = client.embeddings.create(model=EMBED_MODEL, input=sentences)
vectors = [item.embedding for item in response.data]

print(f"{len(vectors)} vectors, {len(vectors[0])} numbers each")
print(f"tokens used: {response.usage.total_tokens}")
print(f"\nfirst 8 numbers of vector 0:\n{[round(v, 4) for v in vectors[0][:8]]}")

2,560 numbers per vector. That is fixed for this model and you cannot ask for fewer,
which Step 4 comes back to.

The numbers themselves mean nothing on their own. They are only useful compared to
other vectors from the same model.

## Step 3: Compare meaning

Cosine similarity measures the angle between two vectors. It runs from -1 to 1, and
higher means more alike. The first two sentences say the same thing in different
words, and the third is unrelated.

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("cat vs feline  ", round(cosine(vectors[0], vectors[1]), 4), " <- same meaning")
print("cat vs revenue ", round(cosine(vectors[0], vectors[2]), 4), " <- unrelated")
print("cat vs itself  ", round(cosine(vectors[0], vectors[0]), 4))

Not one shared word between the first two sentences, and the score is still high.
That is what embeddings buy you over keyword search.

One caution that matters more than it looks. Similarity is a **ranking** signal. A
high score means two pieces of text are about the same thing. It does not mean one
answers the other, and it never means an answer is correct.

## Step 4: The two limits

Two constraints decide how you chunk documents later. Both are worth hitting once,
now, in a notebook rather than in production.

**The `dimensions` parameter does not work here.** Some embedding models let you
request shorter vectors. This one cannot, and it tells you why.

In [ ]:
try:
    client.embeddings.create(model=EMBED_MODEL, input=["test"], dimensions=256)
    print("accepted, which is unexpected")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc)[:260])

A clear error that names the real reason. Matryoshka representation is a training
technique that makes the front of a vector useful on its own, so it can be truncated.
This model was not trained that way, and slicing its vectors by hand would quietly
degrade your search quality. If you need smaller vectors, pick a different model
rather than cutting these.

**There is a token budget per request.** It covers everything you send in one call,
so a long document has to be split before you embed it.

In [ ]:
# One oversized input, to see the limit the endpoint enforces.
try:
    client.embeddings.create(model=EMBED_MODEL, input=["word " * 20000])
    print("accepted")
except Exception as exc:
    print(str(exc)[:260])

That number is the reason chunking exists. The next notebook splits documents to fit
inside it, which is also better for retrieval: a vector averaged over an entire
document is too vague to rank well against a specific question.

## Validation

In [ ]:
assert len(vectors) == 3, "expected three vectors"
assert len(vectors[0]) == 2560, f"expected 2560 dimensions, got {len(vectors[0])}"
assert cosine(vectors[0], vectors[1]) > cosine(vectors[0], vectors[2]), \
    "paraphrase should score higher than an unrelated sentence"
print("Embeddings basics complete.")
print("You can embed text, compare meaning, and you know the two limits.")

## Continue

Next: **[2_build_a_rag_index.ipynb](2_build_a_rag_index.ipynb)**, which turns this
into a search index over real documents and uses it to answer questions.